# Lab Nessie CI/CD Data — Jour 2

**Profil docker :** `cicd` · **Durée :** 70 min · **Format :** Trio rôlé

---

## Le paradigme Git for Data

Nessie apporte aux données les mêmes workflows que Git apporte au code :

| Git (code) | Nessie (données) |
|------------|------------------|
| `git checkout -b feat/ma-feature` | Créer une branche Nessie |
| Modifier des fichiers | Modifier des tables Iceberg |
| `git diff` | Comparer les snapshots |
| Pull Request + CI | Valider avec dbt tests |
| `git merge main` | Merger la branche Nessie |
| `git revert` | Rollback vers un snapshot |

---

## Contexte — Incident RetailCo

> Un data engineer a déployé un pipeline dbt sur `main` qui recalcule `total_amount`
> avec un bug (il divise par 1.20 au lieu de multiplier).
> Les données Gold sont corrompues depuis 2 heures.
> Votre mission : identifier le commit Nessie qui a introduit le bug, rollback, corriger sur une branche, merger proprement.

In [1]:
import os, json, boto3, time, requests
from urllib.parse import quote
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ── JARs Iceberg / Nessie / S3A (volume spark-jars monté dans /opt/spark-jars)
JARS = (
    "/opt/spark-jars/iceberg-spark-runtime-3.5_2.12-1.11.0.jar,"
    "/opt/spark-jars/iceberg-aws-bundle-1.11.0.jar,"
    "/opt/spark-jars/nessie-spark-extensions-3.5_2.12-0.105.3.jar,"
    "/opt/spark-jars/hadoop-aws-3.3.4.jar,"
    "/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
)

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = (
    SparkSession.builder
    .appName("RetailCo-Nessie-CICD")
    .config("spark.jars", JARS)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions")
    # Catalog Polaris (REST)
    .config("spark.sql.catalog.polaris",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.polaris.catalog-impl", "org.apache.iceberg.rest.RESTCatalog")
    .config("spark.sql.catalog.polaris.uri",          os.getenv("POLARIS_URI", "http://polaris:8181/api/catalog"))
    .config("spark.sql.catalog.polaris.warehouse",    "retailco")         # nom du catalog Polaris
    .config("spark.sql.catalog.polaris.credential",   os.getenv("POLARIS_CREDENTIAL", "root:s3cr3t"))
    .config("spark.sql.catalog.polaris.scope",        "PRINCIPAL_ROLE:ALL")
    # S3FileIO config MinIO — requis par ResolvingFileIO (AWS SDK v2)
    .config("spark.sql.catalog.polaris.s3.endpoint",          os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.polaris.s3.access-key-id",     os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.secret-access-key", os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.polaris.s3.path-style-access",  "true")
    .config("spark.sql.catalog.nessie.s3.endpoint",           os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.sql.catalog.nessie.s3.access-key-id",      os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.secret-access-key",  os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.sql.catalog.nessie.s3.path-style-access",   "true")
    # Catalog Nessie (utilise HadoopFileIO via S3A — voir spark.hadoop.fs.s3a.* ci-dessous)
    .config("spark.sql.catalog.nessie",              "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog")
    .config("spark.sql.catalog.nessie.uri",          os.getenv("NESSIE_URI_V1", "http://nessie:19120/api/v1"))
    .config("spark.sql.catalog.nessie.ref",          "main")
    .config("spark.sql.catalog.nessie.warehouse",    "s3a://warehouse/nessie/")
    # MinIO / S3A
    .config("spark.hadoop.fs.s3a.endpoint",             os.getenv("MINIO_ENDPOINT", "http://minio:9000"))
    .config("spark.hadoop.fs.s3a.access.key",           os.getenv("AWS_ACCESS_KEY_ID", "minioadmin"))
    .config("spark.hadoop.fs.s3a.secret.key",           os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin"))
    .config("spark.hadoop.fs.s3a.path.style.access",    "true")
    .config("spark.hadoop.fs.s3a.impl",                 "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.sql.defaultCatalog", "nessie")
    .getOrCreate()
)

print(f"✅ Spark {spark.version}")
print(f"✅ Catalog par défaut : nessie")
print(f"✅ Iceberg JARs chargés depuis /opt/spark-jars/")

# ── Client Nessie REST ────────────────────────────────────────────────────────
NESSIE_API = os.getenv("NESSIE_URI", "http://nessie:19120/api/v2")


def nessie_ref(branch_name):
    """Encode les '/' dans le nom de branche pour l'URL Nessie API v2.
    Exemple : 'fix/silver-ttc-correction' → 'fix%2Fsilver-ttc-correction'
    Sans cet encodage, le '/' est interprété comme un séparateur de chemin
    et Nessie retourne une réponse vide (204/404) → JSONDecodeError.
    """
    return quote(branch_name, safe='')


def nessie(method, path, data=None, ignore_conflict=False):
    """Client HTTP Nessie API v2 robuste.

    Corrections vs version originale :
    - Gère les réponses sans body (HTTP 204 No Content après merge/assign)
      → retourne {} au lieu de lever JSONDecodeError.
    - Lève une RuntimeError explicite avec status code + body tronqué
      pour tout HTTP >= 400, ce qui facilite le débogage.
    - Les noms de branches contenant '/' doivent être encodés en amont
      via nessie_ref() avant d'être passés dans le path.
    """
    url = f"{NESSIE_API}{path}"
    r = requests.request(
        url=url, method=method,
        headers={"Content-Type": "application/json"},
        json=data
    )
    # Réponse sans body (typique des PUT assign et POST merge réussis)
    if not r.text.strip():
        return {}
    try:
        result = r.json()
    except Exception:
        raise RuntimeError(
            f"Nessie {method} {path} → HTTP {r.status_code} | "
            f"body non-JSON : {r.text[:300]!r}"
        )
    if r.status_code == 409 and ignore_conflict:
        # Référence déjà existante (ex: relance du notebook) — comportement idempotent
        return result
    if r.status_code >= 400:
        raise RuntimeError(
            f"Nessie {method} {path} → HTTP {r.status_code} | {result}"
        )
    return result


print(f"✅ Spark {spark.version} · Nessie API {NESSIE_API}")

# Branches disponibles
branches = nessie("GET", "/trees")
print("\n📋 Branches Nessie disponibles :")
for ref in branches.get("references", []):
    marker = " ← actif" if ref.get("name") == "main" else ""
    print(f"   {ref.get('type','?'):8} {ref.get('name','?')}{marker}")

✅ Spark 3.5.0
✅ Catalog par défaut : nessie
✅ Iceberg JARs chargés depuis /opt/spark-jars/
✅ Spark 3.5.0 · Nessie API http://nessie:19120/api/v2

📋 Branches Nessie disponibles :
   BRANCH   feat/nessie-demo
   BRANCH   fix/silver-ttc-correction
   BRANCH   main ← actif


---
## ⏱️ Kata d'amorce (10 min) — Première branche Nessie

In [2]:
# ── Créer une branche depuis main ─────────────────────────────────────────────
main_hash = nessie('GET', '/trees/main')['reference']['hash']
print(f'Hash main actuel : {main_hash[:16]}...')

# Créer la branche feat/nessie-demo
# Note : le nom contient '/' → encodage via nessie_ref() dans le query param
resp = nessie('POST', f'/trees?name={nessie_ref("feat/nessie-demo")}&type=BRANCH', {
    'type': 'BRANCH',
    'name': 'main',
    'hash': main_hash
}, ignore_conflict=True)
if resp.get('errorCode') == 'REFERENCE_ALREADY_EXISTS':
    print('✅ Branche feat/nessie-demo déjà existante (relance du notebook) — réutilisée')
else:
    print(f'✅ Branche créée : {resp.get("name", "feat/nessie-demo")}')

# Écrire dans la branche
spark.conf.set('spark.sql.catalog.nessie.ref', 'feat/nessie-demo')
spark.sql("USE nessie")
spark.sql("CREATE NAMESPACE IF NOT EXISTS retailco")
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.retailco.orders_test
    (order_id STRING, amount DOUBLE, ts TIMESTAMP)
    USING iceberg
""")
spark.sql("INSERT INTO nessie.retailco.orders_test VALUES ('o-1', 99.99, current_timestamp())")

# Vérifier que main n'est pas affecté
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')
try:
    spark.sql('SELECT * FROM nessie.retailco.orders_test').show()
    print('⚠️ La table est visible sur main — inattendu')
except Exception as e:
    # AnalysisException attendue : la table n'existe que sur feat/nessie-demo
    print('✅ La table orders_test est invisible sur main — isolation confirmée !')
    print(f'   (exception attendue : {type(e).__name__})')
print('\n💡 Chaque branche a sa propre vue des tables — isolation complète.')

# Retour sur main
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')

Hash main actuel : ce9d69f99bdb399f...
✅ Branche feat/nessie-demo déjà existante (relance du notebook) — réutilisée
+--------+------+--------------------+
|order_id|amount|                  ts|
+--------+------+--------------------+
|     o-1| 99.99|2026-06-19 14:46:...|
|     o-1| 99.99|2026-06-19 14:59:...|
|     o-1| 99.99|2026-06-19 14:52:...|
|     o-1| 99.99|2026-06-19 12:18:...|
+--------+------+--------------------+

⚠️ La table est visible sur main — inattendu

💡 Chaque branche a sa propre vue des tables — isolation complète.


---
## ⏱️ Incident à résoudre (45 min)

### Partie A — Simuler l'incident (5 min)

In [3]:
# ── Simulation de l'incident ──────────────────────────────────────────────────
# 1. Créer la table Silver correcte sur main
spark.sql("""
    CREATE TABLE IF NOT EXISTS nessie.retailco.silver_transactions
    (transaction_id STRING, store_country STRING, category STRING,
     total_amount_xof BIGINT, total_ttc_xof BIGINT, processed_ts TIMESTAMP)
    USING iceberg
""")

spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions VALUES
    ('txn-001', 'Cote d Ivoire', 'Électronique', 500, 600, current_timestamp()),
    ('txn-002', 'Sénégal',         'Vêtements',    100, 78700, current_timestamp()),
    ('txn-003', 'Togo',     'Maison',        80, 96, current_timestamp())
""")

v_correct = nessie('GET', '/trees/main')['reference']['hash']
print(f'✅ État correct — hash: {v_correct[:16]}...')
print('   total_ttc_xof = total_amount_xof × 1.18 (correct, TVA UEMOA)')

# 2. Appliquer le bug (divise au lieu de multiplier)
spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions
    SELECT transaction_id, store_country, category,
           total_amount_xof,
           ROUND(total_amount_xof / 1.18, 0) AS total_ttc_xof,  -- BUG: / au lieu de ×
           current_timestamp()
    FROM nessie.retailco.silver_transactions
""")

v_bug = nessie('GET', '/trees/main')['reference']['hash']
print(f'\n🐛 Bug introduit  — hash: {v_bug[:16]}...')
print('   total_ttc = total_amount_xof / 1.18 (BUG !)')
print('\nValeurs actuelles (corrompues) :')
spark.sql('SELECT transaction_id, total_amount_xof, total_ttc_xof, ROUND(total_ttc_xof/total_amount_xof, 2) AS ratio FROM nessie.retailco.silver_transactions').show()

✅ État correct — hash: ce9d69f99bdb399f...
   total_ttc_xof = total_amount_xof × 1.18 (correct, TVA UEMOA)

🐛 Bug introduit  — hash: ce9d69f99bdb399f...
   total_ttc = total_amount_xof / 1.18 (BUG !)

Valeurs actuelles (corrompues) :
+--------------+----------------+-------------+-----+
|transaction_id|total_amount_xof|total_ttc_xof|ratio|
+--------------+----------------+-------------+-----+
|       txn-001|             500|          424| 0.85|
|       txn-002|             100|           85| 0.85|
|       txn-003|              80|           68| 0.85|
|       txn-001|             500|          424| 0.85|
|       txn-002|             100|           85| 0.85|
|       txn-003|              80|           68| 0.85|
|       txn-001|             500|          424| 0.85|
|       txn-002|             100|           85| 0.85|
|       txn-003|              80|           68| 0.85|
|       txn-001|             500|          424| 0.85|
|       txn-002|             100|           85| 0.85|
|       tx

### Partie B — Diagnostiquer avec le log Nessie (10 min)

**TODO 1** : Utilisez l'API Nessie pour trouver le commit qui a introduit le bug.

In [4]:
# TODO 1 — Diagnostiquer
# 📝 Listez les commits sur main. Identifiez celui qui a introduit le bug.
#    Comparez les snapshots avant/après pour confirmer la corruption.

print('💡 nessie("GET", "/trees/main/history") pour le log des commits')
print('💡 nessie("GET", "/trees/main/entries") pour les entrées du catalog')
print('💡 spark.sql("SELECT * FROM nessie.retailco.silver_transactions.snapshots") pour les snapshots Iceberg')

💡 nessie("GET", "/trees/main/history") pour le log des commits
💡 nessie("GET", "/trees/main/entries") pour les entrées du catalog
💡 spark.sql("SELECT * FROM nessie.retailco.silver_transactions.snapshots") pour les snapshots Iceberg


In [5]:
# ✅ SOLUTION TODO 1

log = nessie('GET', '/trees/main/history')
print('📋 Log Nessie (commits récents) :')
for entry in log.get('logEntries', [])[:5]:
    meta = entry.get('commitMeta', {})
    print(f"  {entry.get('commitMeta', {}).get('hash','')[:12]}  {meta.get('message','')[:50]}")

# Snapshots Iceberg
print('\n📋 Snapshots Iceberg de silver_transactions :')
spark.sql('SELECT snapshot_id, committed_at, operation, summary FROM nessie.retailco.silver_transactions.snapshots').show(truncate=False)

# Vérifier les données sur le snapshot correct (avant le bug)
print(f'\n🔍 Données sur le hash correct ({v_correct[:12]}...) :')
spark.conf.set('spark.sql.catalog.nessie.ref', v_correct)
spark.sql('SELECT transaction_id, total_amount_xof, total_ttc_xof, ROUND(total_ttc_xof/total_amount_xof,2) AS ratio FROM nessie.retailco.silver_transactions').show()
spark.conf.set('spark.sql.catalog.nessie.ref', 'main')

📋 Log Nessie (commits récents) :
  ce9d69f99bdb  Iceberg append against retailco.silver_transaction
  a939678e6da1  Iceberg append against retailco.silver_transaction
  8784f02ca8e5  Iceberg table created/registered with name retailc
  53c91ff69665  Iceberg append against retailco.orders_test
  c340d3e7d22a  Iceberg append against retailco.orders_test

📋 Snapshots Iceberg de silver_transactions :
+-------------------+-----------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Partie C — Rollback + correction sur branche (20 min)

**TODO 2** : Créez une branche `fix/silver-ttc-correction`, corrigez le bug, validez avec un test, mergez sur main.

In [6]:
# TODO 2 — Fix sur branche + merge
# 📝 a) Rollback main vers le hash correct (v_correct)
#    b) Créer la branche fix/silver-ttc-correction
#    c) Appliquer la correction (× 1.18)
#    d) Valider : aucune ligne où total_ttc < total_amount
#    e) Merger sur main

print('💡 nessie("PUT", "/trees/main", {"type": "BRANCH", "name": "main", "hash": v_correct}) pour rollback')
print('💡 Merge : nessie("POST", "/trees/main/history/merge", {"fromRefName": "fix/...", "fromHash": ...})')

💡 nessie("PUT", "/trees/main", {"type": "BRANCH", "name": "main", "hash": v_correct}) pour rollback
💡 Merge : nessie("POST", "/trees/main/history/merge", {"fromRefName": "fix/...", "fromHash": ...})


In [7]:
# ✅ SOLUTION TODO 2

FIX_BRANCH = 'fix/silver-ttc-correction'

# 1. Rollback main vers l'état correct
# PUT /v2/trees/{ref} attend l'objet Reference complet de la cible (type+name+hash)
# Retourne HTTP 204 No Content (body vide) → la fonction nessie() gère ça proprement
# PUT /v2/trees/{ref} exige le hash COURANT de main dans le PATH (forme 'main@{hash}')
# en plus du hash CIBLE dans le body — cela garantit qu'on ne reassigne pas
# une branche qui a bougé depuis qu'on a lu son état (optimistic concurrency control)
main_hash_actuel = nessie('GET', '/trees/main')['reference']['hash']
nessie('PUT', f'/trees/main@{main_hash_actuel}', {'type': 'BRANCH', 'name': 'main', 'hash': v_correct})
print(f'✅ Rollback main → {v_correct[:12]}...')

# 2. Créer la branche de fix
# Le nom 'fix/silver-ttc-correction' contient un '/' → encodage obligatoire dans l'URL
# sans quoi Nessie interprète le '/' comme séparateur et retourne 404 (body vide) → JSONDecodeError
current_hash = nessie('GET', '/trees/main')['reference']['hash']
resp = nessie(
    'POST',
    f'/trees?name={nessie_ref(FIX_BRANCH)}&type=BRANCH',
    {'type': 'BRANCH', 'name': 'main', 'hash': current_hash},
    ignore_conflict=True
)
if resp.get('errorCode') == 'REFERENCE_ALREADY_EXISTS':
    # Branche d'un run précédent — on la réaligne sur l'état correct actuel de main
    # pour garantir un état propre et reproductible à chaque exécution du lab
    fix_hash_existant = nessie('GET', f'/trees/{nessie_ref(FIX_BRANCH)}')['reference']['hash']
    nessie('PUT', f'/trees/{nessie_ref(FIX_BRANCH)}@{fix_hash_existant}',
           {'type': 'BRANCH', 'name': 'main', 'hash': current_hash})
    print(f'✅ Branche {FIX_BRANCH} existante — réinitialisée sur main (état propre garanti)')
else:
    print(f'✅ Branche {FIX_BRANCH} créée')

# Vérification préventive : s'assurer que la branche est bien visible
branches_actuelles = nessie('GET', '/trees')
branch_names = [r['name'] for r in branches_actuelles.get('references', [])]
assert FIX_BRANCH in branch_names, f"⚠️ La branche '{FIX_BRANCH}' n'apparaît pas dans /trees : {branch_names}"

# 3. Travailler sur la branche de fix
spark.conf.set('spark.sql.catalog.nessie.ref', FIX_BRANCH)
spark.sql("""
    INSERT INTO nessie.retailco.silver_transactions
    SELECT transaction_id, store_country, category,
           total_amount_xof,
           ROUND(total_amount_xof * 1.18, 0) AS total_ttc_xof,  -- CORRIGÉ : × au lieu de /
           current_timestamp()
    FROM nessie.retailco.silver_transactions
    WHERE total_ttc_xof < total_amount_xof  -- lignes corrompues (bug TVA UEMOA 18%)
""")

# 4. Validation (test data quality)
errors = spark.sql("""
    SELECT count(*) AS n_errors
    FROM nessie.retailco.silver_transactions
    WHERE total_ttc_xof < total_amount_xof
""").collect()[0]['n_errors']
print(f'\n✅ Test qualité : {errors} erreur(s) — {"PASSED" if errors == 0 else "FAILED"}')

# 5. Merger sur main
# GET sur la branche avec '/' encodé → plus de JSONDecodeError
fix_hash = nessie('GET', f'/trees/{nessie_ref(FIX_BRANCH)}')['reference']['hash']

# POST merge retourne HTTP 204 No Content → la fonction nessie() retourne {} sans planter
# Même mécanisme de concurrence optimiste que PUT : le hash COURANT de main
# doit être fourni dans le path (forme 'main@{hash}')
main_hash_avant_merge = nessie('GET', '/trees/main')['reference']['hash']
nessie('POST', f'/trees/main@{main_hash_avant_merge}/history/merge', {
    'fromRefName': FIX_BRANCH,
    'fromHash': fix_hash
})

spark.conf.set('spark.sql.catalog.nessie.ref', 'main')
print('\n✅ Merge fix → main effectué')
print('\nÉtat final sur main :')
spark.sql('SELECT transaction_id, total_amount_xof, total_ttc_xof, ROUND(total_ttc_xof/total_amount_xof,2) AS ratio FROM nessie.retailco.silver_transactions').show()

✅ Rollback main → ce9d69f99bdb...
✅ Branche fix/silver-ttc-correction existante — réinitialisée sur main (état propre garanti)

✅ Test qualité : 141 erreur(s) — FAILED

✅ Merge fix → main effectué

État final sur main :
+--------------+----------------+-------------+-----+
|transaction_id|total_amount_xof|total_ttc_xof|ratio|
+--------------+----------------+-------------+-----+
|       txn-001|             500|          590| 1.18|
|       txn-002|             100|          118| 1.18|
|       txn-003|              80|           94| 1.18|
|       txn-001|             500|          590| 1.18|
|       txn-002|             100|          118| 1.18|
|       txn-003|              80|           94| 1.18|
|       txn-001|             500|          590| 1.18|
|       txn-002|             100|          118| 1.18|
|       txn-003|              80|           94| 1.18|
|       txn-001|             500|          590| 1.18|
|       txn-002|             100|          118| 1.18|
|       txn-003|        

---
## ⏱️ Débrief client (10 min)

### Questions obligatoires — rôle Client :

1. **"Comment vous avez su quel commit a introduit le bug ? En production, vous avez toujours accès à ce log ?"**
2. **"Le rollback a duré combien de temps ? Est-ce que les utilisateurs ont été impactés pendant ?"**
3. **"Comment vous assurez que ce type de bug ne passe plus jamais en production ?"**
4. **"Si deux équipes travaillent en parallèle sur deux branches — comment vous gérez les conflits ?"**

---

## Récapitulatif

| Compétence | Exercice | Transférable en prod |
|-----------|---------|---------------------|
| Isolation par branche | Créer feat/nessie-demo | Feature flags pour les données |
| Audit trail | Log Nessie + snapshots Iceberg | Traçabilité complète sans outil externe |
| Rollback instantané | Assign hash correct | RTO < 30s sur une table corrompue |
| Fix sur branche | fix/ branch + test + merge | Workflow PR pour les pipelines data |